In [4]:
import pandas as pd
import numpy as np
import os

In [5]:
# directory containing experiment folders
base_path = "./data"

# helper function to read CSV files
def load_csv(filepath):
    return pd.read_csv(filepath, header=None).values

def find_csv_files(folder_path):
    """Try to find CSV files in folder or in a Refreshrate subdirectory"""
    trans_file = os.path.join(folder_path, "translation_vector.csv")
    quat_file = os.path.join(folder_path, "rotation_quaternion.csv")
    
    # Check directly in folder
    if os.path.exists(trans_file) and os.path.exists(quat_file):
        return trans_file, quat_file
    
    # Check in Refreshrate* subdirectory
    try:
        subdirs = [d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d)) and d.startswith("Refreshrate")]
        if subdirs:
            subdir_path = os.path.join(folder_path, subdirs[0])
            trans_file = os.path.join(subdir_path, "translation_vector.csv")
            quat_file = os.path.join(subdir_path, "rotation_quaternion.csv")
            if os.path.exists(trans_file) and os.path.exists(quat_file):
                return trans_file, quat_file
    except:
        pass
    
    return None, None

# data structure: {experiment_group: {calibration_folder: {translation, quaternion}}}
experiment_groups = {}

# iterate over each subdirectory of ./data (each is an experiment group)
for exp_group in sorted(os.listdir(base_path)):
    exp_group_path = os.path.join(base_path, exp_group)
    if not os.path.isdir(exp_group_path):
        continue
    
    experiment_groups[exp_group] = {}
    
    # find all Calibration folders within this experiment group
    for root, dirs, files in os.walk(exp_group_path):
        for d in dirs:
            if d.endswith("Calibration"):
                folder_path = os.path.join(root, d)
                trans_file, quat_file = find_csv_files(folder_path)
                
                if trans_file and quat_file:
                    experiment_groups[exp_group][d] = {
                        "translation": load_csv(trans_file),
                        "quaternion": load_csv(quat_file),
                    }

# display summary
print("Loaded experiment groups:")
for group, experiments in experiment_groups.items():
    print(f"  {group}: {len(experiments)} Calibration folders")

Loaded experiment groups:
  experiment_1Hz: 0 Calibration folders


In [ ]:
# Process each experiment group separately
for group_name, experiments in experiment_groups.items():
    print(f"\n{'='*50}")
    print(f"Experiment Group: {group_name}")
    print(f"{'='*50}")
    
    if not experiments:
        print("No Calibration folders found")
        continue
    
    # list of translation and rotation data for this group
    trans_list = []
    rot_list = []
    
    # populate lists
    for exp_name, data in experiments.items():
        trans_list.append(data["translation"])
        rot_list.append(data["quaternion"])
    
    # compute mean and stddev for translations
    trans_mean = np.mean(trans_list, axis=0)
    rot_mean = np.mean(rot_list, axis=0)
    trans_stddev = np.std(trans_list, axis=0)
    rot_stddev = np.std(rot_list, axis=0)
    
    # compute magnitudes
    translation_distance = np.linalg.norm(trans_mean)
    rotation_degree = np.linalg.norm(rot_mean) * (180.0 / np.pi)
    
    # compute mean of absolute sequential differences
    trans_magnitudes = [np.linalg.norm(t) for t in trans_list]
    trans_seq_diff_mean = 0.0
    if len(trans_magnitudes) > 1:
        trans_seq_diffs = [abs(trans_magnitudes[i+1] - trans_magnitudes[i]) for i in range(len(trans_magnitudes)-1)]
        trans_seq_diff_mean = np.mean(trans_seq_diffs)
    
    # display results for this group
    print(f"Number of Calibrations: {len(experiments)}")
    print(f"Mean Translation Magnitude (cm): {translation_distance:.4f}")
    print(f"Mean Rotation Magnitude (degrees): {rotation_degree:.4f}")
    print(f"Mean Sequential Translation Difference (cm): {trans_seq_diff_mean:.4f}")


Experiment Group: experiment_1Hz
No Calibration folders found


: 